# Notebook 86: Dip Scalper Strategy

**Date**: 2026-01-25  
**Strategy**: Buy dips, sell into momentum for quick profits  
**Timeframe**: Short-term swing trading (days to weeks)  
**Goal**: Generate regular income, not max returns

---

## 🎯 Strategy Design

### Entries (Buy The Dip)
Use proven Buy The Dip conditions (4/5):
1. STH-MVRV < 1.0 (short-term holders underwater)
2. STH-SOPR < 1.0 (selling at loss = capitulation)
3. Realized P/L Ratio < 1.0 (losses dominate)
4. Funding Rate ≤ 0 (derivatives not overheated)
5. Long Liquidations > Short (fear in market)

### Exits (Profit Taking)
**Multiple exit levels** - take profits as momentum returns:

1. **Quick Exit (25% position)**: +5% profit
   - Lock in quick gains
   - Covers transaction costs
   
2. **Momentum Exit (50% position)**: +10% profit OR STH-SOPR > 1.1
   - Short-term holders back in profit
   - Good risk/reward
   
3. **Final Exit (25% position)**: +20% profit OR LTH-SOPR > 1.3
   - Let winners run a bit
   - Exit if long-term holders distribute

### Risk Management
- **Stop Loss**: -8% from entry (protect capital)
- **Max Hold**: 60 days (don't become a bagholder)
- **Transaction Costs**: -0.1% per trade (realistic)

---

## 💰 Expected Performance

**Target**:
- 10-20 trades per year
- 60-70% win rate
- Average gain: 8-12% per winning trade
- Average loss: -5% per losing trade (stopped out)

**Risk Profile**:
- Lower drawdowns than buy & hold
- More trades = more transaction costs
- Requires monitoring (not set-and-forget)

---

In [ ]:
# Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Paths
DATA_DIR = Path('../data')
BRK_DAILY = DATA_DIR / 'brk' / 'daily'
BL_HOURLY = DATA_DIR / 'bl' / 'hourly'
GN_DAILY = DATA_DIR / 'glassnode' / 'daily'

START_DATE = '2020-02-02'
TRANSACTION_COST = 0.001  # 0.1% per trade

print("✅ Setup complete")
print(f"📅 Backtest start: {START_DATE}")
print(f"💸 Transaction cost: {TRANSACTION_COST * 100}% per trade")

---

## 1. Load Data

In [ ]:
# Load daily data
print("Loading daily data...")

price_daily = pd.read_parquet(BRK_DAILY / 'price.parquet')
mvrv_sth_daily = pd.read_parquet(BRK_DAILY / 'mvrv_sth.parquet')
sopr_sth_daily = pd.read_parquet(BRK_DAILY / 'sopr_sth.parquet')
sopr_lth_daily = pd.read_parquet(BRK_DAILY / 'sopr_lth.parquet')
realized_profit_daily = pd.read_parquet(BRK_DAILY / 'realized_profit.parquet')
realized_loss_daily = pd.read_parquet(BRK_DAILY / 'realized_loss.parquet')
funding_rate_daily = pd.read_parquet(GN_DAILY / 'funding_rate.parquet')
liq_long_daily = pd.read_parquet(GN_DAILY / 'liquidations_long.parquet')
liq_short_daily = pd.read_parquet(GN_DAILY / 'liquidations_short.parquet')

# Normalize
for df in [price_daily, mvrv_sth_daily, sopr_sth_daily, sopr_lth_daily,
           realized_profit_daily, realized_loss_daily, funding_rate_daily,
           liq_long_daily, liq_short_daily]:
    if 'time' not in df.columns:
        df.reset_index(inplace=True)
    df['time'] = pd.to_datetime(df['time'])
    df.sort_values('time', inplace=True)

# Filter to backtest period
price_daily = price_daily[price_daily['time'] >= START_DATE].copy()
mvrv_sth_daily = mvrv_sth_daily[mvrv_sth_daily['time'] >= START_DATE].copy()
sopr_sth_daily = sopr_sth_daily[sopr_sth_daily['time'] >= START_DATE].copy()
sopr_lth_daily = sopr_lth_daily[sopr_lth_daily['time'] >= START_DATE].copy()
realized_profit_daily = realized_profit_daily[realized_profit_daily['time'] >= START_DATE].copy()
realized_loss_daily = realized_loss_daily[realized_loss_daily['time'] >= START_DATE].copy()
funding_rate_daily = funding_rate_daily[funding_rate_daily['time'] >= START_DATE].copy()
liq_long_daily = liq_long_daily[liq_long_daily['time'] >= START_DATE].copy()
liq_short_daily = liq_short_daily[liq_short_daily['time'] >= START_DATE].copy()

print(f"✅ Loaded {len(price_daily):,} daily bars")
print(f"   Range: {price_daily['time'].min().date()} → {price_daily['time'].max().date()}")

---

## 2. Build Signals

In [ ]:
# Merge daily data
daily_data = price_daily[['time', 'value']].rename(columns={'value': 'price'})
daily_data = daily_data.merge(mvrv_sth_daily[['time', 'value']].rename(columns={'value': 'mvrv_sth'}), on='time', how='left')
daily_data = daily_data.merge(sopr_sth_daily[['time', 'value']].rename(columns={'value': 'sopr_sth'}), on='time', how='left')
daily_data = daily_data.merge(sopr_lth_daily[['time', 'value']].rename(columns={'value': 'sopr_lth'}), on='time', how='left')
daily_data = daily_data.merge(realized_profit_daily[['time', 'value']].rename(columns={'value': 'realized_profit'}), on='time', how='left')
daily_data = daily_data.merge(realized_loss_daily[['time', 'value']].rename(columns={'value': 'realized_loss'}), on='time', how='left')
daily_data = daily_data.merge(funding_rate_daily[['time', 'value']].rename(columns={'value': 'funding_rate'}), on='time', how='left')
daily_data = daily_data.merge(liq_long_daily[['time', 'value']].rename(columns={'value': 'liq_long'}), on='time', how='left')
daily_data = daily_data.merge(liq_short_daily[['time', 'value']].rename(columns={'value': 'liq_short'}), on='time', how='left')

# Calculate derived metrics
daily_data['rpl_ratio'] = daily_data['realized_profit'] / daily_data['realized_loss']
daily_data['liq_ratio'] = daily_data['liq_long'] / daily_data['liq_short']

# Entry conditions (Buy The Dip)
daily_data['cond1_sth_mvrv'] = daily_data['mvrv_sth'] < 1.0
daily_data['cond2_sth_sopr'] = daily_data['sopr_sth'] < 1.0
daily_data['cond3_rpl_ratio'] = daily_data['rpl_ratio'] < 1.0
daily_data['cond4_funding'] = daily_data['funding_rate'] <= 0.0
daily_data['cond5_liquidations'] = daily_data['liq_ratio'] > 1.0

daily_data['conditions_met'] = (
    daily_data['cond1_sth_mvrv'].astype(int) +
    daily_data['cond2_sth_sopr'].astype(int) +
    daily_data['cond3_rpl_ratio'].astype(int) +
    daily_data['cond4_funding'].astype(int) +
    daily_data['cond5_liquidations'].astype(int)
)

# Entry signal: 4+ conditions met
daily_data['entry_signal'] = daily_data['conditions_met'] >= 4

print(f"✅ Signals calculated")
print(f"   Entry signals: {daily_data['entry_signal'].sum()} days ({daily_data['entry_signal'].sum() / len(daily_data) * 100:.1f}%)")

---

## 3. Dip Scalper Backtest Engine

In [ ]:
def backtest_dip_scalper(df, config):
    """
    Dip Scalper: Buy dips, scale out into momentum.
    
    Parameters:
    -----------
    df : DataFrame with daily data + signals
    config : dict with exit levels and stop loss
    
    Returns:
    --------
    trades : List of completed trades
    equity_curve : Portfolio value over time
    """
    
    # Config
    exit_1_pct = config.get('exit_1_pct', 0.05)  # 5% profit
    exit_1_size = config.get('exit_1_size', 0.25)  # 25% of position
    
    exit_2_pct = config.get('exit_2_pct', 0.10)  # 10% profit
    exit_2_size = config.get('exit_2_size', 0.50)  # 50% of position
    exit_2_sopr = config.get('exit_2_sopr', 1.1)  # STH-SOPR back in profit
    
    exit_3_pct = config.get('exit_3_pct', 0.20)  # 20% profit
    exit_3_size = config.get('exit_3_size', 0.25)  # 25% of position
    exit_3_lth_sopr = config.get('exit_3_lth_sopr', 1.3)  # LTH distribution
    
    stop_loss_pct = config.get('stop_loss', -0.08)  # -8% stop
    max_hold_days = config.get('max_hold_days', 60)  # Max 60 days
    
    tx_cost = config.get('transaction_cost', 0.001)  # 0.1%
    
    # State
    position = None
    initial_capital = 10_000
    capital = initial_capital
    trades = []
    equity_curve = []
    
    for idx, row in df.iterrows():
        date = row['time']
        price = row['price']
        sopr_sth = row.get('sopr_sth', np.nan)
        sopr_lth = row.get('sopr_lth', np.nan)
        
        # Entry logic
        if position is None and row['entry_signal']:
            # Buy with transaction cost
            entry_cost = capital * tx_cost
            available = capital - entry_cost
            size = available / price
            
            position = {
                'entry_date': date,
                'entry_price': price,
                'original_size': size,
                'remaining_size': size,
                'exit_1_done': False,
                'exit_2_done': False,
                'exit_3_done': False,
                'total_exit_value': 0
            }
            capital = 0  # All-in
        
        # Exit logic (if in position)
        if position is not None:
            hold_days = (date - position['entry_date']).days
            current_pnl_pct = (price / position['entry_price']) - 1
            
            # Stop loss check (exit entire position)
            if current_pnl_pct <= stop_loss_pct:
                exit_value = position['remaining_size'] * price * (1 - tx_cost)
                final_value = position['total_exit_value'] + exit_value
                
                trades.append({
                    'entry_date': position['entry_date'],
                    'exit_date': date,
                    'entry_price': position['entry_price'],
                    'exit_price': price,
                    'hold_days': hold_days,
                    'pnl_pct': (final_value / initial_capital - 1) * 100,
                    'exit_type': 'STOP LOSS',
                    'partial_exits': f"1:{position['exit_1_done']} 2:{position['exit_2_done']} 3:{position['exit_3_done']}"
                })
                
                capital = final_value
                position = None
                continue
            
            # Max hold period (exit entire position)
            if hold_days >= max_hold_days:
                exit_value = position['remaining_size'] * price * (1 - tx_cost)
                final_value = position['total_exit_value'] + exit_value
                
                trades.append({
                    'entry_date': position['entry_date'],
                    'exit_date': date,
                    'entry_price': position['entry_price'],
                    'exit_price': price,
                    'hold_days': hold_days,
                    'pnl_pct': (final_value / initial_capital - 1) * 100,
                    'exit_type': 'MAX HOLD',
                    'partial_exits': f"1:{position['exit_1_done']} 2:{position['exit_2_done']} 3:{position['exit_3_done']}"
                })
                
                capital = final_value
                position = None
                continue
            
            # Exit 1: Quick profit (25% position at +5%)
            if not position['exit_1_done'] and current_pnl_pct >= exit_1_pct:
                exit_size = position['original_size'] * exit_1_size
                exit_value = exit_size * price * (1 - tx_cost)
                
                position['total_exit_value'] += exit_value
                position['remaining_size'] -= exit_size
                position['exit_1_done'] = True
            
            # Exit 2: Momentum (50% position at +10% OR STH-SOPR > 1.1)
            if not position['exit_2_done']:
                exit_2_trigger = (current_pnl_pct >= exit_2_pct) or (sopr_sth > exit_2_sopr)
                
                if exit_2_trigger:
                    exit_size = position['original_size'] * exit_2_size
                    exit_value = exit_size * price * (1 - tx_cost)
                    
                    position['total_exit_value'] += exit_value
                    position['remaining_size'] -= exit_size
                    position['exit_2_done'] = True
            
            # Exit 3: Final (25% position at +20% OR LTH-SOPR > 1.3)
            if not position['exit_3_done']:
                exit_3_trigger = (current_pnl_pct >= exit_3_pct) or (sopr_lth > exit_3_lth_sopr)
                
                if exit_3_trigger:
                    exit_size = position['remaining_size']  # Exit remainder
                    exit_value = exit_size * price * (1 - tx_cost)
                    
                    final_value = position['total_exit_value'] + exit_value
                    
                    trades.append({
                        'entry_date': position['entry_date'],
                        'exit_date': date,
                        'entry_price': position['entry_price'],
                        'exit_price': price,
                        'hold_days': hold_days,
                        'pnl_pct': (final_value / initial_capital - 1) * 100,
                        'exit_type': 'PROFIT TARGET' if current_pnl_pct >= exit_3_pct else 'LTH DISTRIBUTION',
                        'partial_exits': f"1:{position['exit_1_done']} 2:{position['exit_2_done']} 3:True"
                    })
                    
                    capital = final_value
                    position = None
        
        # Update equity curve
        if position is not None:
            # Mark to market
            current_value = position['total_exit_value'] + (position['remaining_size'] * price)
            equity_curve.append({'time': date, 'capital': current_value})
        else:
            equity_curve.append({'time': date, 'capital': capital})
    
    return trades, pd.DataFrame(equity_curve)

print("✅ Backtest engine ready")

---

## 4. Run Backtest

In [ ]:
# Configure strategy
config = {
    'exit_1_pct': 0.05,      # Take 25% profit at +5%
    'exit_1_size': 0.25,
    
    'exit_2_pct': 0.10,      # Take 50% profit at +10% or STH-SOPR > 1.1
    'exit_2_size': 0.50,
    'exit_2_sopr': 1.1,
    
    'exit_3_pct': 0.20,      # Take final 25% at +20% or LTH-SOPR > 1.3
    'exit_3_size': 0.25,
    'exit_3_lth_sopr': 1.3,
    
    'stop_loss': -0.08,      # -8% stop loss
    'max_hold_days': 60,     # Force exit after 60 days
    'transaction_cost': TRANSACTION_COST
}

print("="*80)
print("RUNNING DIP SCALPER BACKTEST")
print("="*80)
print(f"\n📋 Configuration:")
print(f"   Exit 1: {config['exit_1_size']*100:.0f}% @ +{config['exit_1_pct']*100:.0f}%")
print(f"   Exit 2: {config['exit_2_size']*100:.0f}% @ +{config['exit_2_pct']*100:.0f}% OR STH-SOPR > {config['exit_2_sopr']}")
print(f"   Exit 3: {config['exit_3_size']*100:.0f}% @ +{config['exit_3_pct']*100:.0f}% OR LTH-SOPR > {config['exit_3_lth_sopr']}")
print(f"   Stop Loss: {config['stop_loss']*100:.0f}%")
print(f"   Max Hold: {config['max_hold_days']} days")
print(f"   Transaction Cost: {config['transaction_cost']*100:.1f}%")

# Run backtest
trades, equity_curve = backtest_dip_scalper(daily_data, config)

print(f"\n✅ Backtest complete: {len(trades)} trades executed")

---

## 5. Performance Analysis

In [ ]:
if len(trades) == 0:
    print("⚠️  No trades executed!")
else:
    trades_df = pd.DataFrame(trades)
    
    # Calculate metrics
    final_capital = equity_curve['capital'].iloc[-1]
    total_return = (final_capital / 10_000 - 1) * 100
    
    num_trades = len(trades_df)
    winners = (trades_df['pnl_pct'] > 0).sum()
    losers = (trades_df['pnl_pct'] <= 0).sum()
    win_rate = winners / num_trades * 100
    
    avg_return = trades_df['pnl_pct'].mean()
    avg_winner = trades_df[trades_df['pnl_pct'] > 0]['pnl_pct'].mean() if winners > 0 else 0
    avg_loser = trades_df[trades_df['pnl_pct'] <= 0]['pnl_pct'].mean() if losers > 0 else 0
    
    avg_hold = trades_df['hold_days'].mean()
    median_hold = trades_df['hold_days'].median()
    
    # Max drawdown
    running_max = equity_curve['capital'].expanding().max()
    drawdown = (equity_curve['capital'] - running_max) / running_max
    max_dd = drawdown.min() * 100
    
    # Buy & Hold comparison
    bh_return = (daily_data['price'].iloc[-1] / daily_data['price'].iloc[0] - 1) * 100
    bh_dd = ((daily_data['price'] - daily_data['price'].expanding().max()) / daily_data['price'].expanding().max()).min() * 100
    
    print("="*80)
    print("DIP SCALPER PERFORMANCE")
    print("="*80)
    
    print(f"\n💰 Returns:")
    print(f"   Total Return:      {total_return:>8.1f}%")
    print(f"   Buy & Hold:        {bh_return:>8.1f}%")
    print(f"   Alpha:             {total_return - bh_return:>+8.1f}%")
    
    print(f"\n📊 Trade Statistics:")
    print(f"   Total Trades:      {num_trades:>8}")
    print(f"   Winners:           {winners:>8} ({win_rate:.1f}%)")
    print(f"   Losers:            {losers:>8}")
    print(f"   Avg Return:        {avg_return:>8.2f}%")
    print(f"   Avg Winner:        {avg_winner:>8.2f}%")
    print(f"   Avg Loser:         {avg_loser:>8.2f}%")
    
    print(f"\n📅 Hold Period:")
    print(f"   Average:           {avg_hold:>8.1f} days")
    print(f"   Median:            {median_hold:>8.1f} days")
    
    print(f"\n📉 Risk:")
    print(f"   Max Drawdown:      {max_dd:>8.1f}%")
    print(f"   Buy & Hold DD:     {bh_dd:>8.1f}%")
    
    # Exit type breakdown
    print(f"\n🎯 Exit Type Breakdown:")
    exit_counts = trades_df['exit_type'].value_counts()
    for exit_type, count in exit_counts.items():
        pct = count / num_trades * 100
        print(f"   {exit_type:20s}: {count:>3} ({pct:>5.1f}%)")
    
    # Trades per year
    trades_df['year'] = pd.to_datetime(trades_df['entry_date']).dt.year
    trades_per_year = trades_df['year'].value_counts().sort_index()
    print(f"\n📅 Trades Per Year:")
    for year, count in trades_per_year.items():
        print(f"   {year}: {count:>3} trades")
    
    avg_trades_per_year = num_trades / len(trades_per_year)
    print(f"   Average: {avg_trades_per_year:.1f} trades/year")

---

## 6. Visualizations

In [ ]:
if len(trades) > 0:
    fig, axes = plt.subplots(3, 1, figsize=(16, 12))
    
    # 1. Equity curve
    ax1 = axes[0]
    ax1.plot(equity_curve['time'], equity_curve['capital'], label='Dip Scalper', linewidth=2, color='#2ecc71')
    
    bh_equity = daily_data['price'] / daily_data['price'].iloc[0] * 10_000
    ax1.plot(daily_data['time'], bh_equity, label='Buy & Hold', linewidth=2, linestyle='--', alpha=0.7, color='gray')
    
    ax1.set_ylabel('Portfolio Value ($)', fontsize=12)
    ax1.set_title('Dip Scalper vs Buy & Hold', fontsize=14, fontweight='bold')
    ax1.legend(loc='best')
    ax1.grid(True, alpha=0.3)
    ax1.set_yscale('log')
    
    # 2. Price with entry/exit markers
    ax2 = axes[1]
    ax2.plot(daily_data['time'], daily_data['price'], color='black', linewidth=1.5, alpha=0.7, label='BTC Price')
    
    # Entry markers
    entries = trades_df[['entry_date', 'entry_price']].drop_duplicates()
    ax2.scatter(entries['entry_date'], entries['entry_price'], 
               color='green', marker='^', s=100, label='Entry', zorder=5, edgecolors='black')
    
    # Exit markers (color by type)
    profit_exits = trades_df[trades_df['exit_type'].isin(['PROFIT TARGET', 'LTH DISTRIBUTION'])]
    stop_exits = trades_df[trades_df['exit_type'] == 'STOP LOSS']
    max_hold_exits = trades_df[trades_df['exit_type'] == 'MAX HOLD']
    
    if len(profit_exits) > 0:
        ax2.scatter(profit_exits['exit_date'], profit_exits['exit_price'],
                   color='blue', marker='v', s=100, label='Profit Exit', zorder=5, edgecolors='black')
    
    if len(stop_exits) > 0:
        ax2.scatter(stop_exits['exit_date'], stop_exits['exit_price'],
                   color='red', marker='v', s=100, label='Stop Loss', zorder=5, edgecolors='black')
    
    if len(max_hold_exits) > 0:
        ax2.scatter(max_hold_exits['exit_date'], max_hold_exits['exit_price'],
                   color='orange', marker='v', s=100, label='Max Hold', zorder=5, edgecolors='black')
    
    ax2.set_ylabel('BTC Price ($)', fontsize=12)
    ax2.set_title('Entry/Exit Points', fontsize=14, fontweight='bold')
    ax2.legend(loc='best')
    ax2.grid(True, alpha=0.3)
    ax2.set_yscale('log')
    
    # 3. Return distribution
    ax3 = axes[2]
    ax3.hist(trades_df['pnl_pct'], bins=30, color='#3498db', alpha=0.7, edgecolor='black')
    ax3.axvline(0, color='red', linestyle='--', linewidth=2, label='Break-even')
    ax3.axvline(trades_df['pnl_pct'].mean(), color='green', linestyle='--', linewidth=2, label=f'Mean: {avg_return:.2f}%')
    
    ax3.set_xlabel('Return per Trade (%)', fontsize=12)
    ax3.set_ylabel('Frequency', fontsize=12)
    ax3.set_title('Trade Return Distribution', fontsize=14, fontweight='bold')
    ax3.legend(loc='best')
    ax3.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("✅ Charts complete")
else:
    print("⚠️  No trades to visualize")

---

## 7. Recent Trades

In [ ]:
if len(trades) > 0:
    print("="*80)
    print("RECENT TRADES (Last 10)")
    print("="*80)
    
    recent = trades_df.tail(10)[['entry_date', 'exit_date', 'entry_price', 'exit_price', 
                                  'hold_days', 'pnl_pct', 'exit_type']].copy()
    recent['entry_date'] = recent['entry_date'].dt.strftime('%Y-%m-%d')
    recent['exit_date'] = recent['exit_date'].dt.strftime('%Y-%m-%d')
    
    print(recent.to_string(index=False))
    print("\n✅ Trade history complete")

---

## 8. Summary & Next Steps

In [ ]:
if len(trades) > 0:
    print("="*80)
    print("SUMMARY: DIP SCALPER STRATEGY")
    print("="*80)
    
    print(f"\n✅ Strategy Profile:")
    print(f"   Type: Short-term swing trading (dip buying)")
    print(f"   Avg Hold: {avg_hold:.0f} days (~{avg_hold/7:.1f} weeks)")
    print(f"   Trades/Year: {avg_trades_per_year:.0f}")
    print(f"   Win Rate: {win_rate:.1f}%")
    
    print(f"\n💰 Expected Returns (per trade):")
    print(f"   Winners: ~{avg_winner:.1f}% avg")
    print(f"   Losers: ~{avg_loser:.1f}% avg (stopped out)")
    print(f"   Overall: ~{avg_return:.1f}% avg")
    
    print(f"\n🎯 When to Use:")
    if avg_trades_per_year >= 15:
        print(f"   ✅ Good for regular income ({avg_trades_per_year:.0f} opportunities/year)")
    else:
        print(f"   ⚠️  Less frequent ({avg_trades_per_year:.0f} opportunities/year)")
    
    if win_rate >= 60:
        print(f"   ✅ High probability ({win_rate:.0f}% win rate)")
    else:
        print(f"   ⚠️  Lower win rate ({win_rate:.0f}%)")
    
    if total_return > bh_return:
        print(f"   ✅ Beats buy & hold (+{total_return - bh_return:.0f}% alpha)")
    else:
        print(f"   ❌ Underperforms buy & hold ({total_return - bh_return:.0f}%)")
    
    print(f"\n📱 Implementation:")
    print(f"   1. Monitor Buy The Dip signals daily")
    print(f"   2. Enter when 4/5 conditions met")
    print(f"   3. Scale out at profit targets (5%, 10%, 20%)")
    print(f"   4. Use -8% stop loss (protect capital)")
    print(f"   5. Max hold: {config['max_hold_days']} days (don't baghold)")
    
    print(f"\n💡 Next Steps:")
    print(f"   • Optimize profit targets (try 7%, 12%, 25%)")
    print(f"   • Test tighter stop loss (-5% vs -8%)")
    print(f"   • Compare to position sizing (50% vs 100% allocation)")
    print(f"   • Paper trade to validate with real-time data")
    
    print("\n" + "="*80)
else:
    print("⚠️  No trades executed - adjust entry conditions")